In [19]:
import pandas as pd

# ---------- 1) Customers ----------
customers = pd.read_csv("customers_master.csv")
customers = customers.drop_duplicates()
customers["Region"] = customers["Region"].str.strip().str.title()
customers.to_csv("customers_master_cleaned.csv", index=False)

# ---------- 2) Products ----------
products = pd.read_csv("products_master.csv")
products = products.drop_duplicates()
products["Category"] = products["Category"].str.strip().str.title()
products.to_csv("products_master_cleaned.csv", index=False)

# ---------- 3) Orders + New batch (same cleaning) ----------
def clean_orders(file_name):
    df = pd.read_csv(file_name)
    df = df.drop_duplicates()

    # text columns: remove spaces and unify letters
    df["Order_Status"] = df["Order_Status"].str.strip().str.title()
    df["Payment_Method"] = df["Payment_Method"].str.strip().str.title()

    # numbers
    df["Quantity"] = pd.to_numeric(df["Quantity"].astype(str).str.replace("pcs", "").str.strip(), errors="coerce")
    df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")

    disc = df["Discount"].astype(str).str.strip()
    is_percent = disc.str.contains("%")                      # like 5%
    df["Discount"] = pd.to_numeric(disc.str.replace("%", ""), errors="coerce")
    df.loc[is_percent, "Discount"] = df["Discount"] / 100    # 5% -> 0.05

    # dates
    df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
    df["Ship_Date"] = pd.to_datetime(df["Ship_Date"], errors="coerce")

    # missing customer
    df["Customer_ID"] = df["Customer_ID"].fillna("Unknown")

    # wrong values -> remove the row
    df = df[df["Quantity"] > 0]
    df = df[(df["Discount"] >= 0) & (df["Discount"] <= 1)]
    df = df.dropna(subset=["Order_Date"])
    df = df[~(df["Ship_Date"] < df["Order_Date"])]
    df = df[df["Product_ID"].isin(products["Product_ID"])]
    return df

orders = clean_orders("orders.csv")
orders.to_csv("orders_cleaned.csv", index=False)

batch = clean_orders("new_daily_batch.csv")
batch.to_csv("new_daily_batch_cleaned.csv", index=False)

# ---------- 4) Previous days summary ----------
summary = pd.read_csv("previous_days_summary.csv")
summary = summary.drop_duplicates()
summary.to_csv("previous_days_summary_cleaned.csv", index=False)

print(len(orders), len(batch))

1010 316
